# Milestone 0: Setup

**Before running anything:** Runtime menu -> Change runtime type -> Hardware accelerator: **T4 GPU** -> Save.

Then run each cell from top to bottom (Shift+Enter). Cells marked **CHECKPOINT** are the ones whose output you should look at and paste back to me.

## Step 1: Check the GPU (CHECKPOINT 1)

`nvidia-smi` asks the NVIDIA driver which GPUs exist. You should see **Tesla T4** and about **15360MiB** of memory.

`nvcc --version` prints the version of the CUDA compiler. We will use nvcc to compile every `.cu` file. You should see a line like `Cuda compilation tools, release 12.x`.

In [ ]:
!nvidia-smi
!nvcc --version

## Step 2: Connect Google Drive and create the project folder

Colab's disk is wiped when the session ends. Your Google Drive is not. So **every file of this project lives in a Drive folder**, and we `cd` into it. That way `%%writefile` saves source code straight to Drive and nothing is lost on timeout.

A pop-up will ask you to allow access to Drive: click Allow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
projectFolder = '/content/drive/MyDrive/isp_project'
os.makedirs(projectFolder, exist_ok=True)
os.makedirs(projectFolder + '/data', exist_ok=True)
os.makedirs(projectFolder + '/output', exist_ok=True)

# %cd changes the notebook's working folder, so later cells read and write here.
%cd /content/drive/MyDrive/isp_project
!ls

## Step 3: Put your raw file in the data folder (YOUR ACTION)

1. On your Mac, download one **DNG** from https://raw.pixls.us (pick any camera, then a `.dng` file; 10 to 25 megapixels is ideal).
2. Open drive.google.com, go to **My Drive -> isp_project -> data**, and drag the file in.
3. Run the cell below. It should list your file. (Drive can take ~30 seconds to sync; rerun if it is empty.)

Remember: this raw file is **Bayer sensor data**. Each pixel measured only one colour (red, green or blue), so it is not a black-and-white photo, even though we will display it as grey in Step 6.

In [ ]:
!ls -lh data

## Step 4: Install rawpy

rawpy is a Python wrapper around LibRaw, which understands camera raw formats. We use it **only** to unpack the file; all image processing will be our own C++/CUDA code.

In [ ]:
!pip install -q rawpy

## Step 5: The conversion script

This writes `convert_raw.py` into your Drive project folder. It:
1. Reads the raw file with rawpy.
2. Finds the Bayer pattern (which colour sits at the top-left 2x2 block).
3. Crops 0 or 1 pixel from the top and/or left so the pattern becomes **RGGB**.
4. Saves the pixels as `bayer.bin` (raw 16-bit unsigned numbers, row by row, no header) and the numbers we need as `metadata.txt`.

### Why cropping one pixel fixes the pattern
The Bayer pattern is a 2x2 tile repeated over the whole sensor. Suppose the top-left tile is
```
G R
B G      (called GRBG)
```
If you throw away the first **column**, the image now starts at the R, and the new top-left tile is
```
R G
G B      (RGGB)
```
Because the pattern repeats every 2 pixels, shifting the starting point by one pixel just picks a different corner of the same tile. So:
- **GRBG**: drop the first column
- **GBRG**: drop the first row
- **BGGR**: drop the first row and the first column
- **RGGB**: drop nothing

We also drop a last row/column if needed so width and height are even, which keeps every 2x2 tile complete. Losing one pixel of edge changes nothing visible, and it means our C++/CUDA code only ever needs to handle RGGB.

In [ ]:
%%writefile convert_raw.py
# convert_raw.py
# Usage: python convert_raw.py data/yourfile.dng
# Output: data/bayer.bin and data/metadata.txt

import sys
import numpy
import rawpy


def describeBayerPattern(rawFile):
    # raw_pattern is a 2x2 array of colour indices for the top-left tile.
    # color_desc tells us what each index means, e.g. b'RGBG'.
    # The fourth entry is a second green, so we map it to plain 'G'.
    colourNames = rawFile.color_desc.decode('ascii')
    patternText = ''
    for rowInTile in range(2):
        for columnInTile in range(2):
            colourIndex = rawFile.raw_pattern[rowInTile][columnInTile]
            patternText = patternText + colourNames[colourIndex]
    return patternText


def cropToRGGB(bayerImage, patternText):
    # Decide how many rows/columns to drop from the top-left (see explanation above).
    rowsToDropAtTop = 0
    columnsToDropAtLeft = 0
    if patternText == 'RGGB':
        rowsToDropAtTop = 0
        columnsToDropAtLeft = 0
    elif patternText == 'GRBG':
        columnsToDropAtLeft = 1
    elif patternText == 'GBRG':
        rowsToDropAtTop = 1
    elif patternText == 'BGGR':
        rowsToDropAtTop = 1
        columnsToDropAtLeft = 1
    else:
        print('Unsupported pattern (not a standard Bayer sensor):', patternText)
        sys.exit(1)

    croppedImage = bayerImage[rowsToDropAtTop:, columnsToDropAtLeft:]

    # Make width and height even so every 2x2 tile is complete.
    imageHeight = croppedImage.shape[0]
    imageWidth = croppedImage.shape[1]
    evenHeight = imageHeight - (imageHeight % 2)
    evenWidth = imageWidth - (imageWidth % 2)
    croppedImage = croppedImage[:evenHeight, :evenWidth]

    print('Cropped', rowsToDropAtTop, 'row(s) at top and', columnsToDropAtLeft, 'column(s) at left')
    return croppedImage


def main():
    inputPath = sys.argv[1]
    rawFile = rawpy.imread(inputPath)

    # raw_image_visible excludes the hidden 'masked' border pixels some sensors have.
    bayerImage = rawFile.raw_image_visible.copy()

    patternText = describeBayerPattern(rawFile)
    print('Original Bayer pattern:', patternText)
    print('Original size (width x height):', bayerImage.shape[1], 'x', bayerImage.shape[0])

    bayerImage = cropToRGGB(bayerImage, patternText)
    imageHeight = bayerImage.shape[0]
    imageWidth = bayerImage.shape[1]

    # Black level: the value a pixel reports in total darkness.
    # Some cameras give one per channel; they are almost always equal, so we use the average.
    blackLevelPerChannel = rawFile.black_level_per_channel
    blackLevel = float(numpy.mean(blackLevelPerChannel))

    # White level: the value of a fully saturated pixel.
    whiteLevel = float(rawFile.white_level)

    # White balance multipliers chosen by the camera when the photo was taken: [R, G, B, G2].
    # We divide by green so green = 1.0, which is the usual convention.
    cameraWhiteBalance = rawFile.camera_whitebalance
    greenMultiplier = cameraWhiteBalance[1]
    redGain = cameraWhiteBalance[0] / greenMultiplier
    greenGain = 1.0
    blueGain = cameraWhiteBalance[2] / greenMultiplier

    # Save pixels as little-endian unsigned 16-bit, row by row.
    bayerAsUint16 = numpy.ascontiguousarray(bayerImage, dtype='<u2')
    bayerAsUint16.tofile('data/bayer.bin')

    # Simple 'name value' lines, easy to read from C++ with ifstream >>.
    metadataFile = open('data/metadata.txt', 'w')
    metadataFile.write('width ' + str(imageWidth) + '\n')
    metadataFile.write('height ' + str(imageHeight) + '\n')
    metadataFile.write('black_level ' + str(blackLevel) + '\n')
    metadataFile.write('white_level ' + str(whiteLevel) + '\n')
    metadataFile.write('wb_red ' + str(redGain) + '\n')
    metadataFile.write('wb_green ' + str(greenGain) + '\n')
    metadataFile.write('wb_blue ' + str(blueGain) + '\n')
    metadataFile.close()

    print('Black level per channel:', blackLevelPerChannel)
    print('Pixel value range in file:', int(bayerImage.min()), 'to', int(bayerImage.max()))
    print('Saved data/bayer.bin and data/metadata.txt')
    print(open('data/metadata.txt').read())


main()


## Run the conversion (CHECKPOINT 2)

The line below finds the first `.dng` in `data/` automatically. Expected output: the original pattern, a crop line, black level per channel, the pixel value range, and metadata such as `white_level` (often 4095, 16383 or 65535), and `wb_red`/`wb_blue` usually between 1 and 3.

In [ ]:
import glob
rawFiles = glob.glob('data/*.dng') + glob.glob('data/*.DNG')
print('Using:', rawFiles[0])
!python convert_raw.py "{rawFiles[0]}"
!ls -lh data

## Step 6: View the Bayer data as a greyscale PNG (CHECKPOINT 3)

We read `bayer.bin` back (proving the file is correct), normalise with black/white level, apply a quick gamma so it is not too dark, and save a PNG.

**What you should see:** a recognisable but grey scene. Zoom in on the crop below it: you should see a fine **checkerboard/grid texture**. That texture is the Bayer mosaic: neighbouring pixels measured different colours, so they have different brightness. Demosaicing (stage 2) is what removes it.

In [ ]:
import numpy
from PIL import Image

# Read the metadata file into a dictionary.
metadata = {}
for line in open('data/metadata.txt'):
    parts = line.split()
    metadata[parts[0]] = float(parts[1])

imageWidth = int(metadata['width'])
imageHeight = int(metadata['height'])

bayerValues = numpy.fromfile('data/bayer.bin', dtype='<u2')
bayerImage = bayerValues.reshape(imageHeight, imageWidth).astype(numpy.float32)

normalised = (bayerImage - metadata['black_level']) / (metadata['white_level'] - metadata['black_level'])
normalised = numpy.clip(normalised, 0.0, 1.0)
brightened = normalised ** (1.0 / 2.2)
greyBytes = (brightened * 255.0).astype(numpy.uint8)

Image.fromarray(greyBytes).save('output/bayer_grey.png')

# A 200x200 crop from the centre, enlarged 4x with no smoothing, to show the mosaic.
centreRow = imageHeight // 2
centreColumn = imageWidth // 2
zoomCrop = greyBytes[centreRow:centreRow + 200, centreColumn:centreColumn + 200]
zoomImage = Image.fromarray(zoomCrop).resize((800, 800), Image.NEAREST)
zoomImage.save('output/bayer_zoom.png')

from IPython.display import display
previewImage = Image.fromarray(greyBytes)
previewImage.thumbnail((1000, 1000))
display(previewImage)
display(zoomImage)
print('Saved output/bayer_grey.png and output/bayer_zoom.png to Drive')

## Step 7: Save this notebook to Drive

File menu -> **Save a copy in Drive**. Your code files (`convert_raw.py`, and all future `.cu`/`.cpp` files) are already in `MyDrive/isp_project` because we `cd`'d there.

**Every new session:** set the T4 runtime, then rerun the Step 2 cell (mount + cd) before anything else.